# Predictor identity and operation usage

This executable example uses synthetic input and a default no-network fixture. In explicit live mode it fits the qualified classifier; created synthetic datasets/predictors are retained. It never deletes state. Usage shows workload and lifecycle events, with no monetary rates or inferred prices.

In [ ]:
from support import notebook_client, prepare
client = notebook_client()
columns, query, predictor = prepare(client)

Inspect the immutable fitted identity and reopen it. Metadata export returns pinned server provenance only; names, deletion plans and recipes remain later routes; a predictor ID is not an exported executable model.

In [ ]:
metadata = client.predictor(predictor["id"])
assert metadata["dataset_id"] == predictor["dataset_id"]
assert metadata["model_version"] == predictor["model_version"]
assert metadata["configuration"] == predictor["configuration"]
assert any(p["id"] == predictor["id"] for p in client.predictors())
client.predict(predictor["id"], columns=columns, rows=query.tolist(), probabilities=False)

One logical fit can have queued/running/finished usage records. Count distinct operation IDs, not every event as another fit; preserve attempt/status details. Row/column metadata describes workloads, not charges.

In [ ]:
events = [e for e in client.usage() if e.get("dataset_id") == predictor["dataset_id"]]
fit_events = [e for e in events if e["kind"] == "fit"]
assert fit_events
assert len({e["operation_id"] for e in fit_events}) == 1
assert any(e["status"] == "succeeded" for e in fit_events)
assert all(e["rows"] >= 0 and e["columns"] >= 0 for e in events)
assert all("predictions" not in e and "probabilities" not in e for e in events)
assert {"fit", "predict"} <= {e["kind"] for e in events}
client.close()

Cache eviction and process restart are independent operational checks. Reopening metadata here does not force or prove them. Datasets/predictors persist until explicit authorized deletion; seven-day temporary result access expiry is distinct from physical storage cleanup.